### Imports & Spark Setup

Initialize Spark, import required libraries and set SEC identity for API requests.

In [1]:
# Imports and Spark initialization
from pyspark.sql import functions as F
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import StructType
from delta.tables import DeltaTable

import pandas as pd
from decimal import Decimal
from datetime import datetime, date
from concurrent.futures import ThreadPoolExecutor, as_completed
from edgar import Company, set_identity
import time

spark = SparkSession.builder.getOrCreate()
set_identity("stefan.werner@example.com")

print("Spark session initialized.")


StatementMeta(, 53eef818-ac6d-45df-b203-5ec7e98b213e, 5, Finished, Available, Finished, False)

Spark session initialized.


### Bronze table definition

Define Bronze schema and ensure table exists. 

In [2]:
# Create Bronze table if not exists and load schema
bronze_tbl = "sec_def14a_bronze"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_tbl} (
    ticker STRING,
    cik STRING,
    accession_number STRING,
    filing_date DATE,
    fiscal_year_end DATE,
    company_name STRING,
    form STRING,
    peo_name STRING,
    peo_total_comp DOUBLE,
    peo_actually_paid_comp DOUBLE,
    neo_avg_total_comp DOUBLE,
    neo_avg_actually_paid_comp DOUBLE,
    total_shareholder_return DOUBLE,
    peer_group_tsr DOUBLE,
    net_income DOUBLE,
    company_selected_measure STRING,
    company_selected_measure_value DOUBLE,
    ingestion_ts TIMESTAMP
)
USING delta
""")

bronze_schema: StructType = spark.table(bronze_tbl).schema
print("Bronze table ready.")


StatementMeta(, 53eef818-ac6d-45df-b203-5ec7e98b213e, 6, Finished, Available, Finished, False)

Bronze table ready.


### Helper functions

The helper functions are used during ingestion to ensure consistent types for Delta Lake.

In [3]:
# Helper functions for type normalization and date parsing
def to_float(value):
    """Convert Decimal/str/None to float safely."""
    if value is None:
        return None
    if isinstance(value, Decimal):
        return float(value)
    try:
        return float(value)
    except:
        return None

def to_date(value):
    """Normalize various date formats to Python date."""
    if value is None:
        return None
    if isinstance(value, date):
        return value
    if isinstance(value, pd.Timestamp):
        return value.date()
    if isinstance(value, str):
        for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%m/%d/%Y", "%m/%d/%y"):
            try:
                return datetime.strptime(value, fmt).date()
            except:
                pass
    return None


StatementMeta(, 53eef818-ac6d-45df-b203-5ec7e98b213e, 7, Finished, Available, Finished, False)

### Load dimension table & existing keys

All tickers for which a DEF14A proxy is available are loaded from dimension table for ingestion. We also load the tickers where we already loaded fact data (executive pay data from inline XBRL tags) in previous runs of this notebook to prepare incremental ingestion.

In [4]:
# Load DEF14A tickers from dimension table
df_dim = spark.table("dim_sec_company_def14a").select("ticker", "cik", "company_name")
tickers = [row["ticker"] for row in df_dim.collect()]
print(f"Loaded {len(tickers)} DEF14A-relevant tickers.")

# Load existing Bronze keys for incremental ingestion
try:
    existing = spark.table(bronze_tbl).select("ticker", "fiscal_year_end").collect()
    existing_keys = {(row["ticker"], row["fiscal_year_end"]) for row in existing}
    print(f"Loaded {len(existing_keys)} existing keys.")
except:
    existing_keys = set()
    print("No existing Bronze table found. Full load.")


StatementMeta(, 53eef818-ac6d-45df-b203-5ec7e98b213e, 8, Finished, Available, Finished, False)

Loaded 6102 DEF14A-relevant tickers.
Loaded 17187 existing keys.


### Ticker processing function

This function encapsulates all logic required to ingest a single ticker in a clean, testable unit. It handles SEC API calls with retry logic to ensure robustness against transient network or rate‑limit failures. It parses and merges the executive compensation and pay‑vs‑performance tables into a unified structure, normalizing dates and numeric fields for consistent downstream processing. 
Incremental ingestion is done at the ticker + financial_year_end level (via existing_keys). This was choosen to be able to extend new financial years for later filings. Companies disclose 5 years of data in their exec pay table. That means when a companies 2025 filing is first processed, pay data for 2024, 2023, 2022, 2021 and 2020 is ectracted. Once the company now files their 2026 filing, only pay data for 2025 is added to our table, leaving is with + 1 year of data with each new filing.  

In [5]:
# Function to process a single ticker (fetch, parse, merge SCT + PvP)
def process_ticker(ticker, existing_keys, max_retries=3):
    """
    Fetch latest DEF14A filing for a ticker, extract SCT and PvP tables,
    normalize fields, and return Bronze-ready Row objects.
    """
    rows = []
    for attempt in range(1, max_retries + 1):
        try:
            company = Company(ticker)
            filings = company.get_filings(form="DEF 14A")

            if filings is None or filings.latest() is None:
                return []

            filing = filings.latest()

            try:
                proxy = filing.obj()
            except:
                return []

            exec_df = getattr(proxy, "executive_compensation", pd.DataFrame())
            if exec_df is None or exec_df.empty:
                exec_df = pd.DataFrame(columns=["fiscal_year_end"])
            else:
                exec_df = exec_df.drop(
                    columns=["peo_actually_paid_comp", "neo_avg_actually_paid_comp"],
                    errors="ignore"
                )

            pvp_df = getattr(proxy, "pay_vs_performance", pd.DataFrame())
            if pvp_df is None or pvp_df.empty:
                pvp_df = pd.DataFrame(columns=["fiscal_year_end"])

            merged = exec_df.merge(pvp_df, on="fiscal_year_end", how="outer")

            merged["ticker"] = ticker
            merged["cik"] = proxy.cik
            merged["accession_number"] = filing.accession_number
            merged["filing_date"] = proxy.filing_date
            merged["company_name"] = proxy.company_name
            merged["form"] = proxy.form

            merged["filing_date"] = merged["filing_date"].apply(to_date)
            merged["fiscal_year_end"] = merged["fiscal_year_end"].apply(to_date)

            for _, r in merged.iterrows():
                key = (r["ticker"], r["fiscal_year_end"])
                if key in existing_keys:
                    continue

                rows.append(Row(
                    ticker=r["ticker"],
                    cik=r["cik"],
                    accession_number=r["accession_number"],
                    filing_date=r["filing_date"],
                    fiscal_year_end=r["fiscal_year_end"],
                    company_name=r["company_name"],
                    form=r["form"],
                    peo_name=getattr(proxy, "peo_name", None),
                    peo_total_comp=to_float(r.get("peo_total_comp")),
                    peo_actually_paid_comp=to_float(r.get("peo_actually_paid_comp")),
                    neo_avg_total_comp=to_float(r.get("neo_avg_total_comp")),
                    neo_avg_actually_paid_comp=to_float(r.get("neo_avg_actually_paid_comp")),
                    total_shareholder_return=to_float(r.get("total_shareholder_return")),
                    peer_group_tsr=to_float(r.get("peer_group_tsr")),
                    net_income=to_float(r.get("net_income")),
                    company_selected_measure=getattr(proxy, "company_selected_measure", None),
                    company_selected_measure_value=to_float(r.get("company_selected_measure_value")),
                    ingestion_ts=datetime.utcnow()
                ))

            time.sleep(0.2)
            return rows

        except Exception as e:
            print(f"[WARN] {ticker} attempt {attempt} failed: {e}")
            time.sleep(attempt)

    print(f"[ERROR] {ticker} failed after {max_retries} attempts.")
    return rows


StatementMeta(, 53eef818-ac6d-45df-b203-5ec7e98b213e, 9, Finished, Available, Finished, False)

### Parallel Ingestion

This cell parallelizes ticker ingestion using a ThreadPoolExecutor. Because API calls are I/O‑bound, Python can process many tickers at the same time while others wait for network responses. Each worker thread handles one ticker independently, which significantly reduces total runtime compared to sequential execution. Errors are captured per‑ticker so the overall job continues, and periodic progress logs provide visibility during long runs.

In [6]:
# Parallel ingestion of all tickers
all_rows = []
max_workers = 8

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(process_ticker, t, existing_keys): t for t in tickers}
    for i, future in enumerate(as_completed(futures), start=1):
        t = futures[future]
        try:
            result_rows = future.result()
            all_rows.extend(result_rows)
        except Exception as e:
            print(f"[ERROR] Unhandled exception for {t}: {type(e).__name__}: {e}")

        if i % 50 == 0:
            print(f"[INFO] Processed {i} tickers... collected rows so far: {len(all_rows)}")

print(f"Collected {len(all_rows)} new rows.")


StatementMeta(, 53eef818-ac6d-45df-b203-5ec7e98b213e, 10, Finished, Available, Finished, False)

[INFO] Processed 50 tickers... collected rows so far: 0
[INFO] Processed 100 tickers... collected rows so far: 0
[INFO] Processed 150 tickers... collected rows so far: 0
[INFO] Processed 200 tickers... collected rows so far: 0


LivyCancelFailure: Cancel failed. You can restart the session to interrupt abnormal execution.

### MERGE into Bronze Table

This cell performs an idempotent Delta MERGE to safely write all newly collected rows into the Bronze table. The MERGE ensures that existing (ticker, fiscal_year_end) combinations are updated while new combinations are inserted, preventing duplicates and guaranteeing consistent incremental ingestion. This approach allows the pipeline to be re‑run without risk of double‑loading or corrupting historical data. Using Delta Lake’s ACID guarantees ensures reliable writes even under concurrency or partial failures. This final step completes the Bronze ingestion with clean, deduplicated, production‑ready data.

In [ ]:
# Write Bronze using Delta MERGE (idempotent)
if all_rows:
    df_new = spark.createDataFrame(all_rows, schema=bronze_schema)

    delta_bronze = DeltaTable.forName(spark, bronze_tbl)
    (
        delta_bronze.alias("t")
        .merge(
            df_new.alias("s"),
            "t.ticker = s.ticker AND t.fiscal_year_end = s.fiscal_year_end"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Merged {len(all_rows)} rows into Bronze.")
    display(spark.table(bronze_tbl).limit(20))
else:
    print("No new rows to write.")


StatementMeta(, 53eef818-ac6d-45df-b203-5ec7e98b213e, -1, Cancelled, , Cancelled, True)